# Laborator 3 -> AI


In [1]:
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
from array import array
import os
from PIL import Image
import sys
import time

In [2]:
'''
Authenticate
Authenticates your credentials and creates a client.
'''
subscription_key = os.environ["VISION_KEY"]
endpoint = os.environ["VISION_ENDPOINT"]
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
'''
END - Authenticate
'''

'\nEND - Authenticate\n'

Urmatoarea parte proceseaza efectiv imaginea si sustrage textul.

In [3]:
img = open("test1.png", "rb")
#img = open("test2.jpeg", "rb")

read_response = computervision_client.read_in_stream(
    image=img,
    mode="Printed",
    raw=True
)

operation_id = read_response.headers["Operation-Location"].split("/")[-1]
while True:
    #se continua loop-ul pana cand se termina procesarea
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted','running']:
        break
    time.sleep(1)

#acum suntem siguri ca s-a terminat procesarea (fie cu succes, fie cu eroare)
result = []
result_caract = ""
result_words = []
if read_result.status == 'succeeded':
    for text_result in read_result.analyze_result.read_results:
        for lines in text_result.lines:
            print(lines.text)
            result.append(lines.text)
            result_caract += lines.text
            for word in lines.words:
                result_words.append(word.text.lower())

print(result_caract)
print(result_words)

Google Cloud
Platform
Google CloudPlatform
['google', 'cloud', 'platform']


Subpunctul a) -> Se studiaza calitatea procesului de recunoastere a textului, atat la nivel de caracter, cat si la nivel de cuvant, prin folosirea unei metrici de distanta. Metrica de distanta folosita aici este distanta Levenshtein.

In [4]:
#Evaluarea codului -> definirea performantei

#groundTruth = ["Succes in rezolvarea", "tEMELOR la", "LABORAtoaree de", "Inteligenta Artificiala!"]
groundTruth = ["Google Cloud","Platform"]
newResult = sum(i == j for i, j in zip(result, groundTruth))

In [5]:
noOfCorrectLines = sum(i == j for i, j in zip(result, groundTruth))
print(f"Numarul de linii corecte: {noOfCorrectLines}")

# se calculeaza distanta Levenshtein pe caractere si pe cuvinte
#correct_result_caract = "Succes in rezolvarea"+"tEMELOR la"+ "LABORAtoarele de"+ "Inteligență Artificială!"
correct_result_caract = "Google Cloud"+"Platform"

#correct_result_word = ["succes","in","rezolvarea","temelor","la","laboratoarele","de","inteligență","artificială!"]
correct_result_word = ["google", "cloud","platform"]



def lev(a,b):
    if len(b) == 0:
        return len(a)
    if len(a) == 0:
        return len(b)
    if a[0] == b[0]:
        return lev(a[1:], b[1:])

    return 1+ min(lev(a[1:], b), lev(a, b[1:]),lev(a[1:], b[1:]))


def lev_fast(a, b):
    rows = len(a) + 1
    cols = len(b) + 1
    dist = [[0 for _ in range(cols)] for _ in range(rows)]

    for i in range(1, rows): dist[i][0] = i
    for i in range(1, cols): dist[0][i] = i

    for col in range(1, cols):
        for row in range(1, rows):
            if a[row-1] == b[col-1]:
                cost = 0
            else:
                cost = 1

            dist[row][col] = min(dist[row-1][col] + 1,      # stergere
                                 dist[row][col-1] + 1,      # inserare
                                 dist[row-1][col-1] + cost) # substitutie
    return dist[-1][-1]

dist_caract = lev_fast(correct_result_caract,result_caract)
dist_words = lev_fast(correct_result_word,result_words)

cer = dist_caract / len(correct_result_caract)
wer = dist_words / len(correct_result_word)

accuracy_caract = (1 - min(1,cer))*100
accuracy_word = (1 - min(1,wer))*100
print(f"Distanta Levenshtein pe caractere:{dist_caract}, cer: {cer}, acuratete:{accuracy_caract}")
print(f"Distanta Levenshtein pe cuvinte: {dist_words}, wer: {wer}, acuratete:{accuracy_word}")




Numarul de linii corecte: 2
Distanta Levenshtein pe caractere:0, cer: 0.0, acuratete:100.0
Distanta Levenshtein pe cuvinte: 0, wer: 0.0, acuratete:100.0


Subpunctul b) -> Se studiaza calitatea procesului de recunoastere a textului, atat la nivel de caracter, cat si la nivel de cuvant, prin folosirea a mai multor metrici de distanta.

In [6]:
#performanta/calitatea pe caractere
print("PENTRU CARACTERE")
print(f"Distanta Levenshtein:{dist_caract}, cer: {cer}, acuratete:{accuracy_caract}")

import jellyfish

similitudine_caract = jellyfish.jaro_winkler_similarity(correct_result_caract, result_caract)
distance_caract = 1 - similitudine_caract
print(f"Similitudine Jaro-Winkler: {similitudine_caract:.4f}")
print(f"Distanta Jaro-Winkler: {distance_caract:.4f}")


def hamming_distance(a, b):
    if len(a) != len(b):
        return -1
    count = 0
    for i in range(len(a)):
        if a[i] != b[i]:
            count += 1

    return count

hamming_distance_caract = hamming_distance(correct_result_caract, result_caract)
if hamming_distance_caract > -1:
    cer = hamming_distance_caract / len(correct_result_caract)
    print(f"Hamming distance: {hamming_distance_caract:.4f}; cer: {cer:.4f}")
else:
    print("The correct answer and the result do not have the same number of characters.")


PENTRU CARACTERE
Distanta Levenshtein:0, cer: 0.0, acuratete:100.0
Similitudine Jaro-Winkler: 1.0000
Distanta Jaro-Winkler: 0.0000
Hamming distance: 0.0000; cer: 0.0000


In [7]:
#performanta/calitatea pe cuvinte
print("PENTRU CUVINTE")
print(f"Distanta Levenshtein:{dist_words}, wer: {wer}, acuratete:{accuracy_word}")

import jellyfish

def jaro_distance(s1, s2):
    """calculeaza distanta Jaro dintre 2 stringuri."""
    if not s1 and not s2:
        return 1.0
    if not s1 or not s2:
        return 0.0

    s1, s2 = s1.lower(), s2.lower()
    len1, len2 = len(s1), len(s2)
    match_distance = max(len1, len2) // 2 - 1

    s1_matches = [False] * len1
    s2_matches = [False] * len2

    matches = 0
    transpositions = 0

    # count matches
    for i in range(len1):
        start = max(0, i - match_distance)
        end = min(i + match_distance + 1, len2)
        for j in range(start, end):
            if s2_matches[j]:
                continue
            if s1[i] != s2[j]:
                continue
            s1_matches[i] = True
            s2_matches[j] = True
            matches += 1
            break

    if matches == 0:
        return 0.0

    # count transpositions
    k = 0
    for i in range(len1):
        if not s1_matches[i]:
            continue
        while not s2_matches[k]:
            k += 1
        if s1[i] != s2[k]:
            transpositions += 1
        k += 1

    return (
        (matches / len1) +
        (matches / len2) +
        ((matches - transpositions / 2) / matches)
    ) / 3.0


def jaro_winkler(s1, s2, prefix_scale: float = 0.1) :
    """calculeaza Jaro–Winkler similarity intre 2 stringuri."""
    jaro_sim = jaro_distance(s1, s2)

    # se gaseste lungimea prefixului comun (max 4)
    prefix_len = 0
    for c1, c2 in zip(s1.lower(), s2.lower()):
        if c1 == c2:
            prefix_len += 1
        else:
            break
        if prefix_len == 4:
            break

    return jaro_sim + (prefix_len * prefix_scale * (1 - jaro_sim))


def compare_word_lists(list1, list2):
    """compara 2 liste de cuvinte folosind Jaro–Winkler similarity."""
    if not isinstance(list1, list) or not isinstance(list2, list):
        raise ValueError("Both inputs must be lists of strings.")
    if not all(isinstance(w, str) for w in list1 + list2):
        raise ValueError("All elements in both lists must be strings.")

    results = []
    for w1 in list1:
        row = []
        for w2 in list2:
            row.append(round(jaro_winkler(w1, w2), 4))
        results.append(row)
    return results


def overall_jaro_winkler_score(list1, list2):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    best_scores = []
    for w1 in list1:
        best = 0.0
        for w2 in list2:
            score = jaro_winkler(w1, w2)
            if score > best:
                best = score
        best_scores.append(best)

    return sum(best_scores) / len(best_scores)

similarity_matrix = compare_word_lists(correct_result_word, result_words)

print("Jaro–Winkler similarity matrix:")
for i, row in enumerate(similarity_matrix):
    print(f"{correct_result_word[i]} -> {row}")
print(f"Jaro-Winkler similarity: {overall_jaro_winkler_score(correct_result_word, result_words)}")


hamming_distance_words = hamming_distance(correct_result_word, result_words)
if hamming_distance_words > -1:
    wer = hamming_distance_words / len(correct_result_word)
    print(f"Hamming distance: {hamming_distance_words:.4f}; cer: {wer:.4f}")
else:
    print("The correct answer and the result do not have the same number of words.")


PENTRU CUVINTE
Distanta Levenshtein:0, wer: 0.0, acuratete:100.0
Jaro–Winkler similarity matrix:
google -> [1.0, 0.4556, 0.3611]
cloud -> [0.4556, 1.0, 0.55]
platform -> [0.3611, 0.55, 1.0]
Jaro-Winkler similarity: 1.0
Hamming distance: 0.0000; cer: 0.0000


Problema 2 -> Calitatea localizarii corecte a textului in imagine

Localizare pe fiecare cuvant in parte


In [8]:
# se identifica localizarea obtinuta de algoritm

coord_resulted= []
if read_result.status == 'succeeded':
    for text_result in read_result.analyze_result.read_results:
        for lines in text_result.lines:
            for word in lines.words:
                coord_resulted.append(word.bounding_box)

print(coord_resulted)


# se citeste GroundTruth-ul
import pandas as pd
import json

dataFrame = pd.read_csv('localizare.csv')
data = dataFrame[dataFrame['filename'] == 'test1.png']

# se extrag coordonatele + width + height din json-ul din fisierul csv
def extract_coords(row):
    shape = json.loads(row['region_shape_attributes'])
    return shape['x'], shape['y'], shape['width'], shape['height']

coordonate = [extract_coords(row) for _, row in data.iterrows()]
print(coordonate)

def IoU(lista_coord,lista):
    for index in range(len(lista_coord)):
        lista_results = lista_coord[index]
        lista_real = lista[index]

        width = lista_real[2]
        height = lista_real[3]

        #trebuie determinate colturile extreme (minime si maxime)
        #se determina coordonatele extreme

        #coltul din stanga sus al reuniunii
        xmin1 = min(lista_results[0],lista_real[0])
        ymin1 = min(lista_results[1],lista_real[1])

        #coltul din stanga sus al intersectiei
        xmin2 = max(lista_results[0],lista_real[0])
        ymin2 = max(lista_results[1],lista_real[1])


        #coltul din dr jos al reuniunii
        xmax1 = max(lista_results[4],lista_real[0]+width)
        ymax1 = max(lista_results[5],lista_real[1]+height)

        #coltul din dr jos al intersectiei
        xmax2 = min(lista_results[4],lista_real[0]+width)
        ymax2 = min(lista_results[5],lista_real[1]+height)

        lungime_intersectie = max(0,xmax2 - xmin2)
        inaltime_intersectie =max(0,ymax2 - ymin2)
        arie_intersectie = lungime_intersectie * inaltime_intersectie

        arie_reuniune = max(0,(lista_results[4] - lista_results[0])) * max(0,(lista_results[5] - lista_results[1])) + height * width - arie_intersectie

        return arie_intersectie/arie_reuniune

print(f"Calitate localizare: {IoU(coord_resulted,coordonate)}")

[[175.0, 54.0, 320.0, 53.0, 320.0, 95.0, 175.0, 98.0], [332.0, 53.0, 410.0, 52.0, 409.0, 94.0, 332.0, 95.0], [235.0, 111.0, 341.0, 110.0, 342.0, 149.0, 235.0, 151.0]]
[(173, 42, 148, 64), (336, 51, 82, 42), (232, 112, 117, 40)]
Calitate localizare: 0.6276393581081081


Problema 3 -> Posibilitatea de imbunatatire a recunoasterii textului

O solutie ar fi sa ne legam de preprocesare, mai exact de partea de binarizare. Ideea acestei preprocesari este transformarea imaginii in alb-negru, astfel incat sa faca procesarea textului mai usoara (avand doar alb si negru in poza, fara nuante de gri).

In [9]:
# cazul in care imaginea este fuzzy / inclinata

import cv2

# se incarca imaginea in modul grayscale
img = cv2.imread('test4.jpeg', 0)

# se va aplica adaprive thresholding -> avantaj fata de metoda clasica cu un singur threshold: calculeaza pragul pentru zone mai mici
# ale imaginii -> util pentru cand apar umbre
# param: (imagine, valoare_max, metoda, tip_prag, dimensiune_bloc, constanta -> aka filtrul de zgomot)
binary_img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 11, 5)

cv2.imwrite('test4_procesat.png', binary_img)


True

Partea de procesare efectiva a imaginii cu Azure

In [10]:
# partea de OCR efectiva
#img = open("test4_procesat.png", "rb")

def procesare(text):
    img = open(text, "rb")
    read_response = computervision_client.read_in_stream(
        image=img,
        mode="Printed",
        raw=True
    )

    operation_id = read_response.headers["Operation-Location"].split("/")[-1]
    while True:
        #se continua loop-ul pana cand se termina procesarea
        read_result = computervision_client.get_read_result(operation_id)
        if read_result.status not in ['notStarted','running']:
            break
        time.sleep(1)

    #acum suntem siguri ca s-a terminat procesarea (fie cu succes, fie cu eroare)
    result = []
    if read_result.status == 'succeeded':
        for text_result in read_result.analyze_result.read_results:
            for lines in text_result.lines:
                print(lines.text)
                for word in lines.words:
                    result.append(word.text.lower())

    return result

print("INAINTE de pre-procesare")
result1 = procesare("test4.jpeg")
print()

print("DUPA pre-procesare")
result = procesare("test4_procesat.png")
print()

INAINTE de pre-procesare
Sigur ce sã vinà
la nunta fratolui
são. Nu ar avea de ce
sã lipscasca.

DUPA pre-procesare
Sigur a sa vina
la nunta fratelui
sãce. Nu ar avec de ce
são lipscascão.



Partea de post-procesare -> SpellChecker

In [11]:
from spellchecker import SpellChecker

# s-a initializat fara o limba predefinita, pentru a utiliza fisierul propriu
spell = SpellChecker(language=None)
spell.word_frequency.load_text_file('big_romanian_list.txt')

corrected_result = []

for word in result:
    # se verifica apartenenta in "dictionar"
    if word.lower() not in spell:
        # se cauta cea mai apropiata varianta
        corectie = spell.correction(word)
        corrected_result.append(corectie if corectie else word)
    else:
        corrected_result.append(word)

print("Text original:", " ".join(result))
print("Text corectat:", " ".join(corrected_result))

Text original: sigur a sa vina la nunta fratelui sãce. nu ar avec de ce são lipscascão.
Text corectat: sigur a sa vina la nunta fratelui sece nu ar avea de ce so lipscascão.
